In [4]:
from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

SOFT_PERIOD_CSV = SOFT_DIR / "mcs_tracks_soft_3h_2009_2025_JJASO.csv"

print("SOFT period file exists:", SOFT_PERIOD_CSV.exists())
print(SOFT_PERIOD_CSV)

if SOFT_PERIOD_CSV.exists():
    df = pd.read_csv(SOFT_PERIOD_CSV)
    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    print("Rows:", len(df))
    print("Time range:", df["time"].min(), "to", df["time"].max())
    print("Years:", df["time"].dt.year.min(), "-", df["time"].dt.year.max())
    print("Months:", sorted(df["time"].dt.month.unique()))
    print("Unique tracks:", df["track_id"].nunique())

SOFT period file exists: True
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_tracks_soft_3h_2009_2025_JJASO.csv
Rows: 718030
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Years: 2009 - 2025
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]
Unique tracks: 291786


In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

# ============================================================
# LABEL SOFT MCS DATASET — 2009–2025 JJASO
# exact lead ±1.5h + box2deg + radius300km
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

MCS_CSV = SOFT_DIR / "mcs_tracks_soft_3h_2009_2025_JJASO.csv"

GEN_CSV = Path(
    r"C:\Users\_s2218026\Documents\lab\ibtracs_genesis_sorted.csv"
)

OUT_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2025_JJASO_zhang_exact_plus_sens300.csv"

LEADS = [6, 12, 24, 48]

BOX_DEG = 2.0
RADIUS_KM = 300.0
TIME_TOL_H = 1.5
MONTHS = [6, 7, 8, 9, 10]

TIME_COL_MCS = "time"
LAT_COL_MCS  = "centroid_lat"
LON_COL_MCS  = "centroid_lon"

TIME_COL_GEN = "genesis_time"
LAT_COL_GEN  = "genesis_lat"
LON_COL_GEN  = "genesis_lon"


def lon_diff_deg(lon_a, lon_b):
    d = (lon_a - lon_b + 180.0) % 360.0 - 180.0
    return np.abs(d)


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    return 2.0 * R * np.arcsin(np.sqrt(a))


def normalize_lon_360(lon):
    return lon % 360.0


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
if not MCS_CSV.exists():
    raise FileNotFoundError(f"SOFT period file missing:\n{MCS_CSV}")

if not GEN_CSV.exists():
    raise FileNotFoundError(f"Genesis file missing:\n{GEN_CSV}")

mcs = pd.read_csv(MCS_CSV)
gen = pd.read_csv(GEN_CSV)

mcs[TIME_COL_MCS] = (
    pd.to_datetime(mcs[TIME_COL_MCS], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

gen[TIME_COL_GEN] = (
    pd.to_datetime(gen[TIME_COL_GEN], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

mcs = mcs.dropna(subset=[TIME_COL_MCS, LAT_COL_MCS, LON_COL_MCS]).copy()
gen = gen.dropna(subset=[TIME_COL_GEN, LAT_COL_GEN, LON_COL_GEN]).copy()

mcs[LON_COL_MCS] = normalize_lon_360(mcs[LON_COL_MCS].astype(float))
gen[LON_COL_GEN] = normalize_lon_360(gen[LON_COL_GEN].astype(float))

# thesis period
mcs = mcs[
    mcs[TIME_COL_MCS].dt.year.between(2009, 2025)
    & mcs[TIME_COL_MCS].dt.month.isin(MONTHS)
].copy()

gen = gen[
    gen[TIME_COL_GEN].dt.year.between(2009, 2025)
    & gen[TIME_COL_GEN].dt.month.isin(MONTHS)
].copy()

# keep genesis only within MCS time range
gen = gen[
    (gen[TIME_COL_GEN] >= mcs[TIME_COL_MCS].min())
    & (gen[TIME_COL_GEN] <= mcs[TIME_COL_MCS].max())
].copy()

# WNP buffered filter
gen = gen[
    gen[LAT_COL_GEN].astype(float).between(-2, 32)
    & gen[LON_COL_GEN].astype(float).between(98, 182)
].copy()

mcs = mcs.sort_values(TIME_COL_MCS).reset_index(drop=True)
gen = gen.sort_values(TIME_COL_GEN).reset_index(drop=True)

print("SOFT MCS rows:", len(mcs))
print("Genesis rows after filtering:", len(gen))
print("MCS time range:", mcs[TIME_COL_MCS].min(), "to", mcs[TIME_COL_MCS].max())
print("Genesis time range:", gen[TIME_COL_GEN].min(), "to", gen[TIME_COL_GEN].max())

# ------------------------------------------------------------
# Initialize labels
# ------------------------------------------------------------
for L in LEADS:
    mcs[f"zhang_y_{L}h"] = 0
    mcs[f"sens300_y_{L}h"] = 0
    mcs[f"zhang_match_count_{L}h"] = 0
    mcs[f"sens300_match_count_{L}h"] = 0

# ------------------------------------------------------------
# Genesis-centered exact-lead labeling
# ------------------------------------------------------------
t0 = time.time()

for gi, g in gen.iterrows():
    gtime = g[TIME_COL_GEN]
    glat = float(g[LAT_COL_GEN])
    glon = float(g[LON_COL_GEN])

    for L in LEADS:
        target_time = gtime - pd.Timedelta(hours=L)

        cand = mcs[
            (mcs[TIME_COL_MCS] >= target_time - pd.Timedelta(hours=TIME_TOL_H))
            & (mcs[TIME_COL_MCS] <= target_time + pd.Timedelta(hours=TIME_TOL_H))
        ]

        if cand.empty:
            continue

        dlat = np.abs(cand[LAT_COL_MCS].astype(float).values - glat)
        dlon = lon_diff_deg(cand[LON_COL_MCS].astype(float).values, glon)

        in_box = (dlat <= BOX_DEG) & (dlon <= BOX_DEG)

        dkm = haversine_km(
            cand[LAT_COL_MCS].astype(float).values,
            cand[LON_COL_MCS].astype(float).values,
            np.full(len(cand), glat),
            np.full(len(cand), glon)
        )

        in_300km = dkm <= RADIUS_KM

        box_idx = cand.index[in_box]
        km_idx = cand.index[in_300km]

        if len(box_idx) > 0:
            mcs.loc[box_idx, f"zhang_y_{L}h"] = 1
            mcs.loc[box_idx, f"zhang_match_count_{L}h"] += 1

        if len(km_idx) > 0:
            mcs.loc[km_idx, f"sens300_y_{L}h"] = 1
            mcs.loc[km_idx, f"sens300_match_count_{L}h"] += 1

    if (gi + 1) % 20 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"Genesis events processed: {gi + 1}/{len(gen)} | elapsed {elapsed:.1f} min")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
mcs.to_csv(OUT_CSV, index=False)

print("\n[SAVED SOFT LABELS]")
print(OUT_CSV)

print("\nPositive counts:")
for L in LEADS:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(mcs[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(mcs[f'sens300_y_{L}h'].sum())}"
    )

print("\nRows with multiple genesis matches:")
for L in LEADS:
    print(
        f"{L}h | "
        f"box2deg multi={int((mcs[f'zhang_match_count_{L}h'] > 1).sum())} | "
        f"radius300km multi={int((mcs[f'sens300_match_count_{L}h'] > 1).sum())}"
    )

SOFT MCS rows: 718030
Genesis rows after filtering: 361
MCS time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Genesis time range: 2009-06-13 06:00:00 to 2024-10-24 00:00:00
Genesis events processed: 20/361 | elapsed 0.0 min
Genesis events processed: 40/361 | elapsed 0.0 min
Genesis events processed: 60/361 | elapsed 0.0 min
Genesis events processed: 80/361 | elapsed 0.0 min
Genesis events processed: 100/361 | elapsed 0.0 min
Genesis events processed: 120/361 | elapsed 0.0 min
Genesis events processed: 140/361 | elapsed 0.0 min
Genesis events processed: 160/361 | elapsed 0.0 min
Genesis events processed: 180/361 | elapsed 0.0 min
Genesis events processed: 200/361 | elapsed 0.0 min
Genesis events processed: 220/361 | elapsed 0.0 min
Genesis events processed: 240/361 | elapsed 0.0 min
Genesis events processed: 260/361 | elapsed 0.1 min
Genesis events processed: 280/361 | elapsed 0.1 min
Genesis events processed: 300/361 | elapsed 0.1 min
Genesis events processed: 320/361 | elapsed 0.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

# ============================================================
# LABEL SOFT MCS DATASET — 2009–2025 JJASO
# exact lead ±1.5h + box2deg + radius300km
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

MCS_CSV = SOFT_DIR / "mcs_tracks_soft_3h_2009_2025_JJASO.csv"

GEN_CSV = Path(
    r"C:\Users\_s2218026\Documents\lab\ibtracs_genesis_sorted.csv"
)

OUT_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2025_JJASO_zhang_exact_plus_sens300.csv"

LEADS = [6, 12, 24, 48]

BOX_DEG = 2.0
RADIUS_KM = 300.0
TIME_TOL_H = 1.5
MONTHS = [6, 7, 8, 9, 10]

TIME_COL_MCS = "time"
LAT_COL_MCS  = "centroid_lat"
LON_COL_MCS  = "centroid_lon"

TIME_COL_GEN = "genesis_time"
LAT_COL_GEN  = "genesis_lat"
LON_COL_GEN  = "genesis_lon"


def lon_diff_deg(lon_a, lon_b):
    d = (lon_a - lon_b + 180.0) % 360.0 - 180.0
    return np.abs(d)


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )

    return 2.0 * R * np.arcsin(np.sqrt(a))


def normalize_lon_360(lon):
    return lon % 360.0


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------
if not MCS_CSV.exists():
    raise FileNotFoundError(f"SOFT period file missing:\n{MCS_CSV}")

if not GEN_CSV.exists():
    raise FileNotFoundError(f"Genesis file missing:\n{GEN_CSV}")

mcs = pd.read_csv(MCS_CSV)
gen = pd.read_csv(GEN_CSV)

mcs[TIME_COL_MCS] = (
    pd.to_datetime(mcs[TIME_COL_MCS], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

gen[TIME_COL_GEN] = (
    pd.to_datetime(gen[TIME_COL_GEN], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

mcs = mcs.dropna(subset=[TIME_COL_MCS, LAT_COL_MCS, LON_COL_MCS]).copy()
gen = gen.dropna(subset=[TIME_COL_GEN, LAT_COL_GEN, LON_COL_GEN]).copy()

mcs[LON_COL_MCS] = normalize_lon_360(mcs[LON_COL_MCS].astype(float))
gen[LON_COL_GEN] = normalize_lon_360(gen[LON_COL_GEN].astype(float))

# thesis period
mcs = mcs[
    mcs[TIME_COL_MCS].dt.year.between(2009, 2025)
    & mcs[TIME_COL_MCS].dt.month.isin(MONTHS)
].copy()

gen = gen[
    gen[TIME_COL_GEN].dt.year.between(2009, 2025)
    & gen[TIME_COL_GEN].dt.month.isin(MONTHS)
].copy()

# keep genesis only within MCS time range
gen = gen[
    (gen[TIME_COL_GEN] >= mcs[TIME_COL_MCS].min())
    & (gen[TIME_COL_GEN] <= mcs[TIME_COL_MCS].max())
].copy()

# WNP buffered filter
gen = gen[
    gen[LAT_COL_GEN].astype(float).between(-2, 32)
    & gen[LON_COL_GEN].astype(float).between(98, 182)
].copy()

mcs = mcs.sort_values(TIME_COL_MCS).reset_index(drop=True)
gen = gen.sort_values(TIME_COL_GEN).reset_index(drop=True)

print("SOFT MCS rows:", len(mcs))
print("Genesis rows after filtering:", len(gen))
print("MCS time range:", mcs[TIME_COL_MCS].min(), "to", mcs[TIME_COL_MCS].max())
print("Genesis time range:", gen[TIME_COL_GEN].min(), "to", gen[TIME_COL_GEN].max())

# ------------------------------------------------------------
# Initialize labels
# ------------------------------------------------------------
for L in LEADS:
    mcs[f"zhang_y_{L}h"] = 0
    mcs[f"sens300_y_{L}h"] = 0
    mcs[f"zhang_match_count_{L}h"] = 0
    mcs[f"sens300_match_count_{L}h"] = 0

# ------------------------------------------------------------
# Genesis-centered exact-lead labeling
# ------------------------------------------------------------
t0 = time.time()

for gi, g in gen.iterrows():
    gtime = g[TIME_COL_GEN]
    glat = float(g[LAT_COL_GEN])
    glon = float(g[LON_COL_GEN])

    for L in LEADS:
        target_time = gtime - pd.Timedelta(hours=L)

        cand = mcs[
            (mcs[TIME_COL_MCS] >= target_time - pd.Timedelta(hours=TIME_TOL_H))
            & (mcs[TIME_COL_MCS] <= target_time + pd.Timedelta(hours=TIME_TOL_H))
        ]

        if cand.empty:
            continue

        dlat = np.abs(cand[LAT_COL_MCS].astype(float).values - glat)
        dlon = lon_diff_deg(cand[LON_COL_MCS].astype(float).values, glon)

        in_box = (dlat <= BOX_DEG) & (dlon <= BOX_DEG)

        dkm = haversine_km(
            cand[LAT_COL_MCS].astype(float).values,
            cand[LON_COL_MCS].astype(float).values,
            np.full(len(cand), glat),
            np.full(len(cand), glon)
        )

        in_300km = dkm <= RADIUS_KM

        box_idx = cand.index[in_box]
        km_idx = cand.index[in_300km]

        if len(box_idx) > 0:
            mcs.loc[box_idx, f"zhang_y_{L}h"] = 1
            mcs.loc[box_idx, f"zhang_match_count_{L}h"] += 1

        if len(km_idx) > 0:
            mcs.loc[km_idx, f"sens300_y_{L}h"] = 1
            mcs.loc[km_idx, f"sens300_match_count_{L}h"] += 1

    if (gi + 1) % 20 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"Genesis events processed: {gi + 1}/{len(gen)} | elapsed {elapsed:.1f} min")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
mcs.to_csv(OUT_CSV, index=False)

print("\n[SAVED SOFT LABELS]")
print(OUT_CSV)

print("\nPositive counts:")
for L in LEADS:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(mcs[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(mcs[f'sens300_y_{L}h'].sum())}"
    )

print("\nRows with multiple genesis matches:")
for L in LEADS:
    print(
        f"{L}h | "
        f"box2deg multi={int((mcs[f'zhang_match_count_{L}h'] > 1).sum())} | "
        f"radius300km multi={int((mcs[f'sens300_match_count_{L}h'] > 1).sum())}"
    )

SOFT MCS rows: 718030
Genesis rows after filtering: 361
MCS time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Genesis time range: 2009-06-13 06:00:00 to 2024-10-24 00:00:00
Genesis events processed: 20/361 | elapsed 0.0 min
Genesis events processed: 40/361 | elapsed 0.0 min
Genesis events processed: 60/361 | elapsed 0.0 min
Genesis events processed: 80/361 | elapsed 0.0 min
Genesis events processed: 100/361 | elapsed 0.0 min
Genesis events processed: 120/361 | elapsed 0.0 min
Genesis events processed: 140/361 | elapsed 0.0 min
Genesis events processed: 160/361 | elapsed 0.0 min
Genesis events processed: 180/361 | elapsed 0.0 min
Genesis events processed: 200/361 | elapsed 0.0 min
Genesis events processed: 220/361 | elapsed 0.0 min
Genesis events processed: 240/361 | elapsed 0.0 min
Genesis events processed: 260/361 | elapsed 0.0 min
Genesis events processed: 280/361 | elapsed 0.1 min
Genesis events processed: 300/361 | elapsed 0.1 min
Genesis events processed: 320/361 | elapsed 0.

In [7]:
from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

LABEL_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2025_JJASO_zhang_exact_plus_sens300.csv"

df = pd.read_csv(LABEL_CSV)
df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

print("Rows:", len(df))
print("Time range:", df["time"].min(), "to", df["time"].max())
print("Years:", df["time"].dt.year.min(), "-", df["time"].dt.year.max())
print("Months:", sorted(df["time"].dt.month.unique()))
print("Unique tracks:", df["track_id"].nunique())

print("\nTotal positive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(df[f'zhang_y_{L}h'].sum())} | "
        f"radius300km={int(df[f'sens300_y_{L}h'].sum())}"
    )

# ------------------------------------------------------------
# Positive counts by year
# ------------------------------------------------------------
year_pos_rows = []

for y, g in df.groupby(df["time"].dt.year):
    row = {"year": int(y), "rows": len(g)}

    for L in [6, 12, 24, 48]:
        row[f"box2deg_{L}h"] = int(g[f"zhang_y_{L}h"].sum())
        row[f"radius300km_{L}h"] = int(g[f"sens300_y_{L}h"].sum())

    year_pos_rows.append(row)

year_qc = pd.DataFrame(year_pos_rows)

display(year_qc)

year_qc.to_csv(SOFT_DIR / "qc_soft_label_positive_counts_by_year.csv", index=False)

print("Saved:")
print(SOFT_DIR / "qc_soft_label_positive_counts_by_year.csv")

Rows: 718030
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
Years: 2009 - 2025
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]
Unique tracks: 291786

Total positive counts:
6h | box2deg=157 | radius300km=228
12h | box2deg=139 | radius300km=211
24h | box2deg=112 | radius300km=161
48h | box2deg=82 | radius300km=128


,year,rows,box2deg_6h,radius300km_6h,box2deg_12h,radius300km_12h,box2deg_24h,radius300km_24h,box2deg_48h,radius300km_48h
0,2009,36527,13,16,9,12,5,7,7,10
1,2010,38300,8,10,4,8,2,3,5,7
2,2011,39016,4,8,8,10,5,8,2,4
3,2012,38273,4,4,3,3,2,3,3,6
4,2013,38101,11,17,8,12,8,11,7,10
5,2014,38975,6,7,7,7,5,5,2,3
6,2015,31783,12,15,7,12,8,10,5,6
7,2016,45360,12,15,8,12,9,14,8,11
8,2017,45867,16,19,10,17,8,11,4,6
9,2018,43378,10,15,12,17,8,13,8,15


Saved:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\qc_soft_label_positive_counts_by_year.csv


In [8]:
from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

IN_LABEL = SOFT_DIR / "mcs_labeled_final_soft_2009_2025_JJASO_zhang_exact_plus_sens300.csv"

OUT_LABEL_2024 = SOFT_DIR / "mcs_labeled_final_soft_2009_2024_JJASO_zhang_exact_plus_sens300.csv"

df = pd.read_csv(IN_LABEL)

df["time"] = (
    pd.to_datetime(df["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

df_2024 = df[
    df["time"].dt.year.between(2009, 2024)
    & df["time"].dt.month.isin([6, 7, 8, 9, 10])
].copy()

df_2024.to_csv(OUT_LABEL_2024, index=False)

print("Saved:")
print(OUT_LABEL_2024)

print("\nRows:", len(df_2024))
print("Time range:", df_2024["time"].min(), "to", df_2024["time"].max())
print("Years:", df_2024["time"].dt.year.min(), "-", df_2024["time"].dt.year.max())

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(df_2024[f'zhang_y_{L}h'].sum())} | "
        f"radius300km={int(df_2024[f'sens300_y_{L}h'].sum())}"
    )

Saved:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_labeled_final_soft_2009_2024_JJASO_zhang_exact_plus_sens300.csv

Rows: 670086
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Years: 2009 - 2024

Positive counts:
6h | box2deg=157 | radius300km=228
12h | box2deg=139 | radius300km=211
24h | box2deg=112 | radius300km=161
48h | box2deg=82 | radius300km=128


In [9]:
# ============================================================
# SOFT FEATURE EXTRACTION — 2009–2024 JJASO
#
# Input:
#   exact-lead SOFT labeled MCS file
#
# Output:
#   monthly SOFT feature files
#   full SOFT feature file
#
# Features:
#   vo850_rel, eta850_abs
#   rh600
#   u200, v200, u850, v850, wshear_200_850
#   sst_K, mslp_Pa
#   pi_vmax_ms, pi_pmin_hPa, pi_ifl
#   gpi
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

from tcpyPI.pi import pi

# ============================================================
# CONFIG
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

MCS_LABELED_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2024_JJASO_zhang_exact_plus_sens300.csv"

# ERA5 environmental variables from your older extraction setup
ERA5_CLEAN_DIR = Path(r"C:\Users\_s2218026\Documents\lab\PAST\data_clean")

# ERA5 SST / MSLP / T-Q profiles downloaded for PI
PI_DIR = Path(r"C:\Users\_s2218026\Documents\lab\NEW\data_clean_PI")

FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"
FEATURE_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2024_JJASO_zhang4pt.csv"

ERR_DIR = FEATURE_MONTHLY_DIR / "errors"
ERR_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2009, 2025))
MONTHS = [6, 7, 8, 9, 10]

SAVE_EVERY = 1000

# ============================================================
# ERA5 file keys
# ============================================================

KEY_VO850 = "vorticity_850"
KEY_RH600 = "temperature-relative_humidity_600"
KEY_UV200 = "u_component_of_wind-v_component_of_wind_200"
KEY_UV850 = "u_component_of_wind-v_component_of_wind_850"

KEY_SST   = "sea_surface_temperature"
KEY_MSLP  = "mean_sea_level_pressure"
KEY_TQTAG = "t-q_profiles"

# ============================================================
# Columns
# ============================================================

BASE_COLS = [
    "track_id", "time",
    "centroid_lat", "centroid_lon",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg"
]

LABEL_COLS = [
    "zhang_y_6h", "zhang_y_12h", "zhang_y_24h", "zhang_y_48h",
    "sens300_y_6h", "sens300_y_12h", "sens300_y_24h", "sens300_y_48h",
]

MATCH_COUNT_COLS = [
    "zhang_match_count_6h", "zhang_match_count_12h", "zhang_match_count_24h", "zhang_match_count_48h",
    "sens300_match_count_6h", "sens300_match_count_12h", "sens300_match_count_24h", "sens300_match_count_48h",
]

FEATURE_COLS = [
    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi"
]

BOOKKEEP_COLS = [
    "attempted_env", "attempted_pi", "status_flag"
]

In [10]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def to_naive_timestamp(x):
    ts = pd.Timestamp(x)
    if ts.tzinfo is not None:
        return ts.tz_convert(None)
    return ts


def open_nc(fp: Path) -> xr.Dataset:
    ds = xr.open_dataset(fp, engine="netcdf4")

    # Standardize coordinate names
    rename_map = {}

    if "valid_time" in ds.coords and "time" not in ds.coords:
        rename_map["valid_time"] = "time"

    if "pressure_level" in ds.coords and "level" not in ds.coords:
        rename_map["pressure_level"] = "level"

    if "lat" in ds.coords and "latitude" not in ds.coords:
        rename_map["lat"] = "latitude"

    if "lon" in ds.coords and "longitude" not in ds.coords:
        rename_map["lon"] = "longitude"

    if rename_map:
        ds = ds.rename(rename_map)

    return ds


def snap_time(ds, t):
    times = pd.DatetimeIndex(pd.to_datetime(ds["time"].values)).tz_localize(None)
    t = to_naive_timestamp(t)

    idx = int(np.argmin(np.abs(times - t)))

    return pd.Timestamp(times[idx])


def pick2d(ds: xr.Dataset, var: str, t_snap: pd.Timestamp) -> xr.DataArray:
    da = ds[var].sel(time=t_snap)

    if "level" in da.dims:
        da = da.isel(level=0, drop=True)

    return da


def coriolis_f(lat_deg):
    Omega = 7.292115e-5
    return 2.0 * Omega * np.sin(np.deg2rad(lat_deg))


def normalize_lon_to_ds(lon, ds_lon):
    lon = float(lon)
    ds_lon = np.asarray(ds_lon, dtype=float)

    if ds_lon.min() >= 0 and lon < 0:
        lon = lon % 360.0

    elif ds_lon.max() <= 180 and lon > 180:
        lon = ((lon + 180.0) % 360.0) - 180.0

    return lon


def surrounding_pair(coord_vals, x):
    vals = np.asarray(coord_vals, dtype=float)

    asc = np.all(np.diff(vals) > 0)
    desc = np.all(np.diff(vals) < 0)

    if not (asc or desc):
        raise ValueError("Coordinate array must be monotonic.")

    vals_work = vals[::-1] if desc else vals

    if np.any(np.isclose(vals_work, x)):
        v = vals_work[np.argmin(np.abs(vals_work - x))]
        return float(v), float(v)

    if x <= vals_work[0]:
        return float(vals_work[0]), float(vals_work[0])

    if x >= vals_work[-1]:
        return float(vals_work[-1]), float(vals_work[-1])

    idx_hi = np.searchsorted(vals_work, x, side="right")
    idx_lo = idx_hi - 1

    lo = float(vals_work[idx_lo])
    hi = float(vals_work[idx_hi])

    return lo, hi


def four_surrounding_mean_2d(da2d: xr.DataArray, lat0, lon0):
    lat_vals = da2d["latitude"].values
    lon_vals = da2d["longitude"].values

    lon0 = normalize_lon_to_ds(lon0, lon_vals)

    lat_a, lat_b = surrounding_pair(lat_vals, float(lat0))
    lon_a, lon_b = surrounding_pair(lon_vals, float(lon0))

    lat_sel = np.unique([lat_a, lat_b])
    lon_sel = np.unique([lon_a, lon_b])

    sub = da2d.sel(latitude=lat_sel, longitude=lon_sel)

    val = sub.mean(skipna=True).values

    return float(val) if np.isfinite(val) else np.nan


def four_surrounding_profile_mean(da3d: xr.DataArray, lat0, lon0):
    levels = da3d["level"].values.astype(float)

    prof = np.array(
        [
            four_surrounding_mean_2d(da3d.sel(level=lev), lat0, lon0)
            for lev in levels
        ],
        dtype=float
    )

    return levels, prof


def find_pi_tq_file(y, m):
    fp = PI_DIR / f"era_{y}_{m:02d}_t-q_profiles.nc"

    if not fp.exists():
        raise FileNotFoundError(f"No monthly T/Q file found: {fp}")

    return fp


def compute_pi(levels_hPa, T_K, q_kgkg, sst_K, mslp_Pa):
    """
    tcpyPI expects:
      SSTC  = SST in Celsius
      MSL   = mean sea level pressure in hPa
      P     = pressure levels in hPa
      TC    = temperature profile in Celsius
      R     = mixing ratio in g/kg
    """
    levels = np.asarray(levels_hPa, dtype=float).copy()
    T_K = np.asarray(T_K, dtype=float).copy()
    q = np.asarray(q_kgkg, dtype=float).copy()

    if not np.isfinite(sst_K) or not np.isfinite(mslp_Pa):
        return np.nan, np.nan, np.nan

    good = np.isfinite(levels) & np.isfinite(T_K) & np.isfinite(q)

    levels = levels[good]
    T_K = T_K[good]
    q = q[good]

    if len(levels) < 4:
        return np.nan, np.nan, np.nan

    q = np.clip(q, 1e-8, 0.05)

    if np.any((T_K < 150) | (T_K > 350)):
        return np.nan, np.nan, np.nan

    if np.any((levels < 1) | (levels > 1100)):
        return np.nan, np.nan, np.nan

    SSTC = float(sst_K - 273.15)
    MSL_hPa = float(mslp_Pa / 100.0)

    if not (-5 <= SSTC <= 45):
        return np.nan, np.nan, np.nan

    if not (850 <= MSL_hPa <= 1100):
        return np.nan, np.nan, np.nan

    # tcpyPI expects pressure from high pressure to low pressure
    if levels[0] < levels[-1]:
        levels = levels[::-1]
        T_K = T_K[::-1]
        q = q[::-1]

    _, uniq_idx = np.unique(levels, return_index=True)
    uniq_idx = np.sort(uniq_idx)

    levels = levels[uniq_idx]
    T_K = T_K[uniq_idx]
    q = q[uniq_idx]

    if len(levels) < 4:
        return np.nan, np.nan, np.nan

    TC_C = T_K - 273.15

    # Convert specific humidity to mixing ratio
    r = q / np.maximum(1.0 - q, 1e-8)
    R_gkg = 1000.0 * r
    R_gkg = np.clip(R_gkg, 1e-6, 80.0)

    try:
        VMAX, PMIN, IFL, TO, OTL = pi(SSTC, MSL_hPa, levels, TC_C, R_gkg)
    except Exception:
        return np.nan, np.nan, np.nan

    if not np.isfinite(VMAX):
        return np.nan, np.nan, np.nan

    if not np.isfinite(PMIN):
        PMIN = np.nan

    try:
        IFL = int(IFL)
    except Exception:
        IFL = np.nan

    return float(VMAX), float(PMIN), IFL


def gpi_zhang_en(eta850_abs, rh600_pct, vpot_ms, vshear_ms):
    eta = np.abs(eta850_abs)
    rh = np.clip(rh600_pct, 0, 100)
    vpot = np.maximum(vpot_ms, 0.0)
    shear = np.maximum(vshear_ms, 0.0)

    return (
        ((1e5 * eta) ** 1.5)
        * ((rh / 50.0) ** 3)
        * ((vpot / 70.0) ** 3)
        * ((1.0 + 0.1 * shear) ** (-2.0))
    )

In [11]:
# ============================================================
# ERA5 DATASET CACHES
# ============================================================

env_cache = {}
pi_2d_cache = {}
tq_cache = {}


def env_ds(y, m, key):
    k = (y, m, key)

    if k not in env_cache:
        fp = ERA5_CLEAN_DIR / f"era_{y}_{m:02d}_{key}.nc"

        if not fp.exists():
            raise FileNotFoundError(fp)

        env_cache[k] = open_nc(fp)

    return env_cache[k]


def get_pi_2d_ds(y, m):
    k = (y, m)

    if k not in pi_2d_cache:
        sst_fp = PI_DIR / f"era_{y}_{m:02d}_{KEY_SST}.nc"
        mslp_fp = PI_DIR / f"era_{y}_{m:02d}_{KEY_MSLP}.nc"

        if not sst_fp.exists():
            raise FileNotFoundError(sst_fp)

        if not mslp_fp.exists():
            raise FileNotFoundError(mslp_fp)

        pi_2d_cache[k] = (open_nc(sst_fp), open_nc(mslp_fp))

    return pi_2d_cache[k]


def get_tq_ds(y, m):
    k = (y, m)

    if k not in tq_cache:
        tq_fp = find_pi_tq_file(y, m)
        tq_cache[k] = open_nc(tq_fp)

    return tq_cache[k]


def close_all_caches():
    for cache in [env_cache, pi_2d_cache, tq_cache]:
        for value in cache.values():
            if isinstance(value, tuple):
                for ds in value:
                    try:
                        ds.close()
                    except Exception:
                        pass
            else:
                try:
                    value.close()
                except Exception:
                    pass

        cache.clear()

In [12]:
# ============================================================
# LOAD LABELED SOFT MCS DATA
# ============================================================

if not MCS_LABELED_CSV.exists():
    raise FileNotFoundError(f"Missing labeled SOFT file:\n{MCS_LABELED_CSV}")

mcs_all = pd.read_csv(MCS_LABELED_CSV)

mcs_all["time"] = (
    pd.to_datetime(mcs_all["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

mcs_all = mcs_all.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

mcs_all = mcs_all[
    mcs_all["time"].dt.year.between(YEARS[0], YEARS[-1])
    & mcs_all["time"].dt.month.isin(MONTHS)
].copy()

# Convenience columns used by ML
mcs_all["month"] = mcs_all["time"].dt.month
mcs_all["lat"] = pd.to_numeric(mcs_all["centroid_lat"], errors="coerce")
mcs_all["lon"] = pd.to_numeric(mcs_all["centroid_lon"], errors="coerce")

# Required checks
required = BASE_COLS + LABEL_COLS

missing = [c for c in required if c not in mcs_all.columns]

if missing:
    raise ValueError(f"Missing required columns in labeled SOFT file:\n{missing}")

print("Loaded SOFT labeled data")
print("Rows:", len(mcs_all))
print("Time range:", mcs_all["time"].min(), "to", mcs_all["time"].max())
print("Years:", mcs_all["time"].dt.year.min(), "-", mcs_all["time"].dt.year.max())
print("Months:", sorted(mcs_all["time"].dt.month.unique()))

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(mcs_all[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(mcs_all[f'sens300_y_{L}h'].sum())}"
    )

Loaded SOFT labeled data
Rows: 670086
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Years: 2009 - 2024
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]

Positive counts:
6h | box2deg/zhang=157 | radius300km/sens300=228
12h | box2deg/zhang=139 | radius300km/sens300=211
24h | box2deg/zhang=112 | radius300km/sens300=161
48h | box2deg/zhang=82 | radius300km/sens300=128


In [13]:
# ============================================================
# EXTRACT FEATURES FOR ONE MONTH
# ============================================================

def extract_features_one_month(year, month, overwrite=False):
    ym = f"{year}{month:02d}"

    out_month = FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_{ym}.csv"
    err_month = ERR_DIR / f"errors_soft_{ym}.csv"

    month_src = mcs_all[
        (mcs_all["time"].dt.year == year)
        & (mcs_all["time"].dt.month == month)
    ].copy()

    if len(month_src) == 0:
        print(f"[SKIP] {ym}: no rows")
        return None

    # ------------------------------------------------------------
    # Resume existing monthly file if present
    # ------------------------------------------------------------
    if out_month.exists() and not overwrite:
        print(f"[RESUME/REUSE] {out_month.name}")
        mcs = pd.read_csv(out_month)

        mcs["time"] = (
            pd.to_datetime(mcs["time"], errors="coerce", utc=True)
            .dt.tz_convert(None)
        )

        if len(mcs) != len(month_src):
            print(f"[WARNING] Existing file row count mismatch for {ym}. Rebuilding.")
            mcs = month_src.copy()
        else:
            print("Existing rows:", len(mcs))

    else:
        print(f"[START] {ym}")
        mcs = month_src.copy()

    # ------------------------------------------------------------
    # Ensure feature/bookkeeping columns exist
    # ------------------------------------------------------------
    for c in FEATURE_COLS:
        if c not in mcs.columns:
            mcs[c] = np.nan

    for c in ["attempted_env", "attempted_pi"]:
        if c not in mcs.columns:
            mcs[c] = 0

    if "status_flag" not in mcs.columns:
        mcs["status_flag"] = ""

    if err_month.exists() and not overwrite:
        err_df = pd.read_csv(err_month)
    else:
        err_df = pd.DataFrame(columns=["row_index", "time", "track_id", "error"])

    # ------------------------------------------------------------
    # Rows still needing attempt
    # ------------------------------------------------------------
    todo_idx = mcs.index[
        (mcs["attempted_pi"] != 1)
    ].tolist()

    print(f"\n{ym}")
    print("Rows:", len(mcs))
    print("Rows still to attempt:", len(todo_idx))

    if len(todo_idx) == 0:
        print(f"[DONE/REUSED] {out_month.name}")
        return out_month

    # ------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------
    pbar = tqdm(todo_idx, desc=f"Extract {ym}", unit="row")

    for n, i in enumerate(pbar, start=1):
        r = mcs.loc[i]

        try:
            t0 = pd.Timestamp(r["time"])
            y, m, d = int(t0.year), int(t0.month), int(t0.day)

            lat0 = float(r["centroid_lat"])
            lon0 = float(r["centroid_lon"])

            # ----------------------------------------------------
            # 850-hPa relative vorticity
            # ----------------------------------------------------
            ds_vo = env_ds(y, m, KEY_VO850)
            tv = snap_time(ds_vo, t0)

            vo_rel = four_surrounding_mean_2d(
                pick2d(ds_vo, "vo", tv),
                lat0,
                lon0
            )

            mcs.at[i, "vo850_rel"] = vo_rel

            eta_abs = vo_rel + float(coriolis_f(lat0)) if np.isfinite(vo_rel) else np.nan
            mcs.at[i, "eta850_abs"] = eta_abs

            # ----------------------------------------------------
            # 600-hPa RH
            # ----------------------------------------------------
            ds_rh = env_ds(y, m, KEY_RH600)
            tr = snap_time(ds_rh, t0)

            rh600 = four_surrounding_mean_2d(
                pick2d(ds_rh, "r", tr),
                lat0,
                lon0
            )

            mcs.at[i, "rh600"] = rh600

            # ----------------------------------------------------
            # Wind shear 200–850 hPa
            # ----------------------------------------------------
            ds200 = env_ds(y, m, KEY_UV200)
            ds850 = env_ds(y, m, KEY_UV850)

            t200 = snap_time(ds200, t0)
            t850 = snap_time(ds850, t0)

            u200 = four_surrounding_mean_2d(pick2d(ds200, "u", t200), lat0, lon0)
            v200 = four_surrounding_mean_2d(pick2d(ds200, "v", t200), lat0, lon0)

            u850 = four_surrounding_mean_2d(pick2d(ds850, "u", t850), lat0, lon0)
            v850 = four_surrounding_mean_2d(pick2d(ds850, "v", t850), lat0, lon0)

            if np.all(np.isfinite([u200, v200, u850, v850])):
                shear = np.sqrt((u200 - u850) ** 2 + (v200 - v850) ** 2)
            else:
                shear = np.nan

            mcs.at[i, "u200"] = u200
            mcs.at[i, "v200"] = v200
            mcs.at[i, "u850"] = u850
            mcs.at[i, "v850"] = v850
            mcs.at[i, "wshear_200_850"] = shear

            mcs.at[i, "attempted_env"] = 1

            # ----------------------------------------------------
            # SST / MSLP
            # ----------------------------------------------------
            ds_sst, ds_msl = get_pi_2d_ds(y, m)

            ts = snap_time(ds_sst, t0)
            tm = snap_time(ds_msl, t0)

            sst_var = "sst" if "sst" in ds_sst.data_vars else "sea_surface_temperature"
            msl_var = "msl" if "msl" in ds_msl.data_vars else "mean_sea_level_pressure"

            sst_K = four_surrounding_mean_2d(
                ds_sst[sst_var].sel(time=ts),
                lat0,
                lon0
            )

            mslp_Pa = four_surrounding_mean_2d(
                ds_msl[msl_var].sel(time=tm),
                lat0,
                lon0
            )

            mcs.at[i, "sst_K"] = sst_K
            mcs.at[i, "mslp_Pa"] = mslp_Pa

            if not np.isfinite(sst_K):
                mcs.at[i, "attempted_pi"] = 1
                mcs.at[i, "status_flag"] = "missing_sst"

                if n % SAVE_EVERY == 0:
                    mcs.to_csv(out_month, index=False)
                    err_df.to_csv(err_month, index=False)

                continue

            # ----------------------------------------------------
            # T/Q profiles
            # ----------------------------------------------------
            ds_tq = get_tq_ds(y, m)
            tt = snap_time(ds_tq, t0)

            T = ds_tq["t"].sel(time=tt)
            Q = ds_tq["q"].sel(time=tt)

            levels, T_lev = four_surrounding_profile_mean(T, lat0, lon0)
            _, Q_lev = four_surrounding_profile_mean(Q, lat0, lon0)

            # ----------------------------------------------------
            # Potential intensity
            # ----------------------------------------------------
            vmax, pmin, ifl = compute_pi(
                levels,
                T_lev,
                Q_lev,
                sst_K,
                mslp_Pa
            )

            mcs.at[i, "pi_vmax_ms"] = vmax
            mcs.at[i, "pi_pmin_hPa"] = pmin
            mcs.at[i, "pi_ifl"] = ifl

            # ----------------------------------------------------
            # GPI
            # ----------------------------------------------------
            if np.all(np.isfinite([eta_abs, rh600, shear, vmax])):
                gpi = gpi_zhang_en(eta_abs, rh600, vmax, shear)
                mcs.at[i, "gpi"] = gpi
                mcs.at[i, "status_flag"] = "done"
            else:
                mcs.at[i, "gpi"] = np.nan
                mcs.at[i, "status_flag"] = "pi_or_env_nan"

            mcs.at[i, "attempted_pi"] = 1

        except Exception as e:
            err_df.loc[len(err_df)] = {
                "row_index": i,
                "time": r.get("time", pd.NaT),
                "track_id": r.get("track_id", None),
                "error": str(e),
            }

            mcs.at[i, "attempted_pi"] = 1
            mcs.at[i, "status_flag"] = "error"

        if n % SAVE_EVERY == 0:
            mcs.to_csv(out_month, index=False)
            err_df.to_csv(err_month, index=False)

            done = int((mcs["attempted_pi"] == 1).sum())
            pbar.set_postfix({"attempted": done, "errors": len(err_df)})

    # ------------------------------------------------------------
    # Save final monthly file
    # ------------------------------------------------------------
    mcs.to_csv(out_month, index=False)
    err_df.to_csv(err_month, index=False)

    print("\n[SAVED MONTH]")
    print(out_month)
    print("Rows:", len(mcs))
    print("Errors:", len(err_df))

    print("\nStatus counts:")
    print(mcs["status_flag"].value_counts(dropna=False))

    print("\nNaN fractions:")
    print(mcs[FEATURE_COLS].isna().mean())

    close_all_caches()

    return out_month



In [14]:
# ============================================================
# RUN SOFT FEATURE EXTRACTION FOR 2009–2024 JJASO
# ============================================================

done_files = []

for y in YEARS:
    for m in MONTHS:
        out_fp = extract_features_one_month(
            year=y,
            month=m,
            overwrite=False
        )

        if out_fp is not None:
            done_files.append(out_fp)

print("\nMonthly feature files available:", len(done_files))

[RESUME/REUSE] mcs_features_PI_GPI_soft_200906.csv
Existing rows: 6867

200906
Rows: 6867
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_200906.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_200907.csv
Existing rows: 8647

200907
Rows: 8647
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_200907.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_200908.csv
Existing rows: 7387

200908
Rows: 7387
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_200908.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_200909.csv
Existing rows: 7555

200909
Rows: 7555
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_200909.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_200910.csv
Existing rows: 6071

200910
Rows: 6071
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_200910.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_201006.csv
Existing rows: 6952

201006
Rows: 6952
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_201006.csv
[RESUME/RE

In [15]:
# ============================================================
# EXTEND NEW/SOFT FEATURE EXTRACTION TO 2025 ONLY
# This assumes your helper functions and extract_features_one_month()
# are already defined in the current notebook.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

MCS_LABELED_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2025_JJASO_zhang_exact_plus_sens300.csv"

# Use the SAME monthly folder if your 2009–2024 files are already there.
# The name says 2009_2024, but it can still store 2025 files.
FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"
FEATURE_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

ERR_DIR = FEATURE_MONTHLY_DIR / "errors"
ERR_DIR.mkdir(parents=True, exist_ok=True)

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

YEARS_2025_ONLY = [2025]
MONTHS = [6, 7, 8, 9, 10]

# ------------------------------------------------------------
# Reload full 2009–2025 labeled NEW/SOFT data
# ------------------------------------------------------------
mcs_all = pd.read_csv(MCS_LABELED_CSV)

mcs_all["time"] = (
    pd.to_datetime(mcs_all["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

mcs_all = mcs_all.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

mcs_all = mcs_all[
    mcs_all["time"].dt.year.between(2009, 2025)
    & mcs_all["time"].dt.month.isin(MONTHS)
].copy()

mcs_all["month"] = mcs_all["time"].dt.month
mcs_all["lat"] = pd.to_numeric(mcs_all["centroid_lat"], errors="coerce")
mcs_all["lon"] = pd.to_numeric(mcs_all["centroid_lon"], errors="coerce")

print("Reloaded NEW/SOFT labeled data")
print("Rows:", len(mcs_all))
print("Time range:", mcs_all["time"].min(), "to", mcs_all["time"].max())
print("2025 rows:", len(mcs_all[mcs_all["time"].dt.year == 2025]))

# ------------------------------------------------------------
# Run only 2025 months
# ------------------------------------------------------------
done_2025 = []

for y in YEARS_2025_ONLY:
    for m in MONTHS:
        out_fp = extract_features_one_month(
            year=y,
            month=m,
            overwrite=False
        )

        if out_fp is not None:
            done_2025.append(out_fp)

print("\n2025 monthly feature files available:", len(done_2025))
for fp in done_2025:
    print(fp)

Reloaded NEW/SOFT labeled data
Rows: 718030
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
2025 rows: 47944
[RESUME/REUSE] mcs_features_PI_GPI_soft_202506.csv
Existing rows: 8987

202506
Rows: 8987
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202506.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_202507.csv
Existing rows: 9887

202507
Rows: 9887
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202507.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_202508.csv
Existing rows: 10541

202508
Rows: 10541
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202508.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_202509.csv
Existing rows: 9935

202509
Rows: 9935
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202509.csv
[RESUME/REUSE] mcs_features_PI_GPI_soft_202510.csv
Existing rows: 8594

202510
Rows: 8594
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202510.csv

2025 monthly feature files available: 5
C:\Users\_s22180

In [16]:
# ============================================================
# CONCATENATE NEW/SOFT FULL FEATURE FILE — 2009–2025
# ============================================================

from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

YEARS = list(range(2009, 2026))
MONTHS = [6, 7, 8, 9, 10]

feature_files = [
    FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_{y}{m:02d}.csv"
    for y in YEARS
    for m in MONTHS
]

missing = [fp for fp in feature_files if not fp.exists()]

print("Expected monthly feature files:", len(feature_files))
print("Missing:", len(missing))

if missing:
    print("\nMissing files:")
    for fp in missing:
        print(fp.name)
    raise FileNotFoundError("Some monthly feature files are missing.")

dfs = []

for fp in feature_files:
    print("reading", fp.name)
    df = pd.read_csv(fp, low_memory=False)
    dfs.append(df)

features = pd.concat(dfs, ignore_index=True)

features["time"] = (
    pd.to_datetime(features["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

features = (
    features
    .sort_values(["time", "track_id"])
    .reset_index(drop=True)
)

features.to_csv(OUT_FULL_FEATURE_CSV, index=False)

print("\n[SAVED FULL NEW/SOFT FEATURES]")
print(OUT_FULL_FEATURE_CSV)

print("\nQC:")
print("Rows:", len(features))
print("Time range:", features["time"].min(), "to", features["time"].max())
print("Years:", features["time"].dt.year.min(), "-", features["time"].dt.year.max())
print("Months:", sorted(features["time"].dt.month.unique()))
print("Unique tracks:", features["track_id"].nunique())

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(features[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(features[f'sens300_y_{L}h'].sum())}"
    )

print("\n2025 positive counts:")
f25 = features[features["time"].dt.year == 2025]
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(f25[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(f25[f'sens300_y_{L}h'].sum())}"
    )

Expected monthly feature files: 85
Missing: 0
reading mcs_features_PI_GPI_soft_200906.csv
reading mcs_features_PI_GPI_soft_200907.csv
reading mcs_features_PI_GPI_soft_200908.csv
reading mcs_features_PI_GPI_soft_200909.csv
reading mcs_features_PI_GPI_soft_200910.csv
reading mcs_features_PI_GPI_soft_201006.csv
reading mcs_features_PI_GPI_soft_201007.csv
reading mcs_features_PI_GPI_soft_201008.csv
reading mcs_features_PI_GPI_soft_201009.csv
reading mcs_features_PI_GPI_soft_201010.csv
reading mcs_features_PI_GPI_soft_201106.csv
reading mcs_features_PI_GPI_soft_201107.csv
reading mcs_features_PI_GPI_soft_201108.csv
reading mcs_features_PI_GPI_soft_201109.csv
reading mcs_features_PI_GPI_soft_201110.csv
reading mcs_features_PI_GPI_soft_201206.csv
reading mcs_features_PI_GPI_soft_201207.csv
reading mcs_features_PI_GPI_soft_201208.csv
reading mcs_features_PI_GPI_soft_201209.csv
reading mcs_features_PI_GPI_soft_201210.csv
reading mcs_features_PI_GPI_soft_201306.csv
reading mcs_features_PI_GPI_so

In [17]:
from pathlib import Path
import pandas as pd

# ============================================================
# CONCATENATE FULL SOFT FEATURE FILE — 2009–2024
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2024_JJASO_zhang4pt.csv"

YEARS = list(range(2009, 2025))
MONTHS = [6, 7, 8, 9, 10]

feature_files = [
    FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_{y}{m:02d}.csv"
    for y in YEARS
    for m in MONTHS
]

missing = [fp for fp in feature_files if not fp.exists()]

print("Expected monthly feature files:", len(feature_files))
print("Missing:", len(missing))

if missing:
    print("\nMissing files:")
    for fp in missing:
        print(fp.name)
    raise FileNotFoundError("Some monthly feature files are missing.")

dfs = []

for fp in feature_files:
    print("reading", fp.name)
    df = pd.read_csv(fp, low_memory=False)
    dfs.append(df)

features = pd.concat(dfs, ignore_index=True)

features["time"] = (
    pd.to_datetime(features["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

features = (
    features
    .sort_values(["time", "track_id"])
    .reset_index(drop=True)
)

features.to_csv(OUT_FULL_FEATURE_CSV, index=False)

print("\n[SAVED FULL SOFT FEATURES]")
print(OUT_FULL_FEATURE_CSV)

print("\nQC:")
print("Rows:", len(features))
print("Time range:", features["time"].min(), "to", features["time"].max())
print("Years:", features["time"].dt.year.min(), "-", features["time"].dt.year.max())
print("Months:", sorted(features["time"].dt.month.unique()))
print("Unique tracks:", features["track_id"].nunique())

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(features[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(features[f'sens300_y_{L}h'].sum())}"
    )

print("\nNaN fractions:")
FEATURE_COLS = [
    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi"
]

print(features[FEATURE_COLS].isna().mean())

Expected monthly feature files: 80
Missing: 0
reading mcs_features_PI_GPI_soft_200906.csv
reading mcs_features_PI_GPI_soft_200907.csv
reading mcs_features_PI_GPI_soft_200908.csv
reading mcs_features_PI_GPI_soft_200909.csv
reading mcs_features_PI_GPI_soft_200910.csv
reading mcs_features_PI_GPI_soft_201006.csv
reading mcs_features_PI_GPI_soft_201007.csv
reading mcs_features_PI_GPI_soft_201008.csv
reading mcs_features_PI_GPI_soft_201009.csv
reading mcs_features_PI_GPI_soft_201010.csv
reading mcs_features_PI_GPI_soft_201106.csv
reading mcs_features_PI_GPI_soft_201107.csv
reading mcs_features_PI_GPI_soft_201108.csv
reading mcs_features_PI_GPI_soft_201109.csv
reading mcs_features_PI_GPI_soft_201110.csv
reading mcs_features_PI_GPI_soft_201206.csv
reading mcs_features_PI_GPI_soft_201207.csv
reading mcs_features_PI_GPI_soft_201208.csv
reading mcs_features_PI_GPI_soft_201209.csv
reading mcs_features_PI_GPI_soft_201210.csv
reading mcs_features_PI_GPI_soft_201306.csv
reading mcs_features_PI_GPI_so

In [18]:
# ============================================================
# FINAL CHECK
# ============================================================

feat = pd.read_csv(OUT_FULL_FEATURE_CSV, low_memory=False)

feat["time"] = (
    pd.to_datetime(feat["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

print("Rows:", len(feat))
print("Time range:", feat["time"].min(), "to", feat["time"].max())
print("Years:", feat["time"].dt.year.min(), "-", feat["time"].dt.year.max())
print("Months:", sorted(feat["time"].dt.month.unique()))
print("Unique tracks:", feat["track_id"].nunique())

BASE_COLS = [
    "track_id", "time",
    "centroid_lat", "centroid_lon",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg"
]

LABEL_COLS = [
    "zhang_y_6h", "zhang_y_12h", "zhang_y_24h", "zhang_y_48h",
    "sens300_y_6h", "sens300_y_12h", "sens300_y_24h", "sens300_y_48h",
]

FEATURE_COLS = [
    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi"
]

print("\nMissing key columns:")
required_cols = BASE_COLS + FEATURE_COLS + LABEL_COLS
missing_cols = [c for c in required_cols if c not in feat.columns]
print(missing_cols)

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg/zhang={int(feat[f'zhang_y_{L}h'].sum())} | "
        f"radius300km/sens300={int(feat[f'sens300_y_{L}h'].sum())}"
    )

print("\nNaN fractions:")
print(feat[[
    "eta850_abs", "rh600", "wshear_200_850",
    "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
    "speed_kmh", "direction_deg"
]].isna().mean())

Rows: 670086
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Years: 2009 - 2024
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]
Unique tracks: 272892

Missing key columns:
[]

Positive counts:
6h | box2deg/zhang=157 | radius300km/sens300=228
12h | box2deg/zhang=139 | radius300km/sens300=211
24h | box2deg/zhang=112 | radius300km/sens300=161
48h | box2deg/zhang=82 | radius300km/sens300=128

NaN fractions:
eta850_abs        0.000000
rh600             0.000000
wshear_200_850    0.000000
sst_K             0.177855
mslp_Pa           0.000000
pi_vmax_ms        0.177856
gpi               0.177856
speed_kmh         0.407249
direction_deg     0.407249
dtype: float64


In [19]:
# ============================================================
# BUILD SOFT MASTER TABLE — 2009–2024
# features + exact-lead labels -> master table
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

FEATURES_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2024_JJASO_zhang4pt.csv"

LABELS_CSV = SOFT_DIR / "mcs_labeled_final_soft_2009_2024_JJASO_zhang_exact_plus_sens300.csv"

OUT_MASTER_CSV = SOFT_DIR / "mcs_master_soft_exactlead_ready_2009_2024_JJASO.csv"

DATASET_NAME = "soft"

LEADS = [6, 12, 24, 48]

# ============================================================
# COLUMNS
# ============================================================

FEATURE_COLS = [
    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi"
]

OLD_LABEL_COLS = [
    "zhang_y_6h", "zhang_y_12h", "zhang_y_24h", "zhang_y_48h",
    "sens300_y_6h", "sens300_y_12h", "sens300_y_24h", "sens300_y_48h"
]

NEW_LABEL_COLS = [
    "box2deg_y_6h", "box2deg_y_12h", "box2deg_y_24h", "box2deg_y_48h",
    "radius300km_y_6h", "radius300km_y_12h", "radius300km_y_24h", "radius300km_y_48h"
]

RENAME_LABELS = {
    "zhang_y_6h": "box2deg_y_6h",
    "zhang_y_12h": "box2deg_y_12h",
    "zhang_y_24h": "box2deg_y_24h",
    "zhang_y_48h": "box2deg_y_48h",
    "sens300_y_6h": "radius300km_y_6h",
    "sens300_y_12h": "radius300km_y_12h",
    "sens300_y_24h": "radius300km_y_24h",
    "sens300_y_48h": "radius300km_y_48h",
}

BASE_COLS = [
    "track_id", "time",
    "centroid_lon", "centroid_lat",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg"
]

MASTER_COLS = [
    "track_id", "time", "month", "lon", "lat",
    "centroid_lon", "centroid_lat",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",

    "vo850_rel", "eta850_abs",
    "rh600",
    "u200", "v200", "u850", "v850", "wshear_200_850",
    "sst_K", "mslp_Pa",
    "pi_vmax_ms", "pi_pmin_hPa", "pi_ifl",
    "gpi",

    "box2deg_y_6h", "box2deg_y_12h", "box2deg_y_24h", "box2deg_y_48h",
    "radius300km_y_6h", "radius300km_y_12h", "radius300km_y_24h", "radius300km_y_48h",
]

# ============================================================
# BUILD MASTER
# ============================================================

def build_soft_master_table(
    features_csv=FEATURES_CSV,
    labels_csv=LABELS_CSV,
    out_master_csv=OUT_MASTER_CSV,
):
    features_csv = Path(features_csv)
    labels_csv = Path(labels_csv)
    out_master_csv = Path(out_master_csv)

    print("\n" + "=" * 70)
    print("BUILD MASTER TABLE: SOFT")
    print("=" * 70)

    if not features_csv.exists():
        raise FileNotFoundError(f"Feature file not found:\n{features_csv}")

    if not labels_csv.exists():
        raise FileNotFoundError(f"Label file not found:\n{labels_csv}")

    feat = pd.read_csv(features_csv)
    lab = pd.read_csv(labels_csv)

    feat["time"] = (
        pd.to_datetime(feat["time"], errors="coerce", utc=True)
        .dt.tz_convert(None)
    )

    lab["time"] = (
        pd.to_datetime(lab["time"], errors="coerce", utc=True)
        .dt.tz_convert(None)
    )

    print("Feature rows:", len(feat))
    print("Label rows:", len(lab))

    print("Feature time range:", feat["time"].min(), "to", feat["time"].max())
    print("Label time range:", lab["time"].min(), "to", lab["time"].max())

    # ------------------------------------------------------------
    # Check columns
    # ------------------------------------------------------------
    missing_feat_cols = [c for c in BASE_COLS + FEATURE_COLS if c not in feat.columns]
    missing_label_cols = [
        c for c in ["track_id", "time", "centroid_lon", "centroid_lat"] + OLD_LABEL_COLS
        if c not in lab.columns
    ]

    if missing_feat_cols:
        raise ValueError(f"Missing feature columns:\n{missing_feat_cols}")

    if missing_label_cols:
        raise ValueError(f"Missing label columns:\n{missing_label_cols}")

    # ------------------------------------------------------------
    # Remove any label columns already inside features
    # Then merge exact-lead labels from label file
    # ------------------------------------------------------------
    feat = feat.drop(
        columns=[c for c in OLD_LABEL_COLS + NEW_LABEL_COLS if c in feat.columns],
        errors="ignore"
    )

    merge_keys = ["track_id", "time", "centroid_lon", "centroid_lat"]

    lab_small = (
        lab[merge_keys + OLD_LABEL_COLS]
        .drop_duplicates(subset=merge_keys)
        .copy()
    )

    lab_small = lab_small.rename(columns=RENAME_LABELS)

    df = feat.merge(
        lab_small,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )

    # ------------------------------------------------------------
    # Add convenience columns
    # ------------------------------------------------------------
    df["month"] = df["time"].dt.month
    df["lon"] = df["centroid_lon"]
    df["lat"] = df["centroid_lat"]

    # ------------------------------------------------------------
    # Validate labels after merge
    # ------------------------------------------------------------
    for c in NEW_LABEL_COLS:
        if c not in df.columns:
            raise ValueError(f"Missing label after merge: {c}")

        n_missing = df[c].isna().sum()
        if n_missing > 0:
            raise ValueError(f"{c} has {n_missing} missing values after merge.")

        df[c] = df[c].astype(int)

    missing_after_merge = [c for c in MASTER_COLS if c not in df.columns]

    if missing_after_merge:
        raise ValueError(f"Missing columns after merge:\n{missing_after_merge}")

    master = df[MASTER_COLS].copy()

    master = (
        master
        .drop_duplicates(subset=["track_id", "time", "centroid_lon", "centroid_lat"])
        .sort_values(["time", "track_id"])
        .reset_index(drop=True)
    )

    out_master_csv.parent.mkdir(parents=True, exist_ok=True)
    master.to_csv(out_master_csv, index=False)

    print("\n[SAVED MASTER]")
    print(out_master_csv)

    print("\nQC:")
    print("Shape:", master.shape)
    print("Time range:", master["time"].min(), "to", master["time"].max())
    print("Years:", master["time"].dt.year.min(), "-", master["time"].dt.year.max())
    print("Months:", sorted(master["time"].dt.month.unique()))
    print("Unique tracks:", master["track_id"].nunique())

    print("\nPositive counts:")
    for L in LEADS:
        print(
            f"{L}h | "
            f"box2deg={int(master[f'box2deg_y_{L}h'].sum())} | "
            f"radius300km={int(master[f'radius300km_y_{L}h'].sum())}"
        )

    print("\nNaN fractions in key predictors:")
    key_predictors = [
        "eta850_abs", "rh600", "wshear_200_850",
        "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
        "speed_kmh", "direction_deg"
    ]

    print(master[key_predictors].isna().mean())

    return master


master_soft = build_soft_master_table()


BUILD MASTER TABLE: SOFT
Feature rows: 670086
Label rows: 670086
Feature time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Label time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00

[SAVED MASTER]
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_master_soft_exactlead_ready_2009_2024_JJASO.csv

QC:
Shape: (670086, 34)
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Years: 2009 - 2024
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]
Unique tracks: 272892

Positive counts:
6h | box2deg=157 | radius300km=228
12h | box2deg=139 | radius300km=211
24h | box2deg=112 | radius300km=161
48h | box2deg=82 | radius300km=128

NaN fractions in key predictors:
eta850_abs        0.000000
rh600             0.000000
wshear_200_850    0.000000
sst_K             0.177855
mslp_Pa           0.000000
pi_vmax_ms        0.177856
gpi               0.177856
speed_kmh         0.407249
direction_deg     0.407249
dtype: float64


In [20]:
from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

SOFT_MASTER_CSV = SOFT_DIR / "mcs_master_soft_exactlead_ready_2009_2024_JJASO.csv"

df = pd.read_csv(SOFT_MASTER_CSV)
df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

print("Rows:", len(df))
print("Time range:", df["time"].min(), "to", df["time"].max())
print("Years:", df["time"].dt.year.min(), "-", df["time"].dt.year.max())
print("Months:", sorted(df["time"].dt.month.unique()))
print("Unique tracks:", df["track_id"].nunique())

print("\nPositive counts:")
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(df[f'box2deg_y_{L}h'].sum())} | "
        f"radius300km={int(df[f'radius300km_y_{L}h'].sum())}"
    )

print("\nNaN fractions:")
print(df[[
    "eta850_abs", "rh600", "wshear_200_850",
    "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
    "speed_kmh", "direction_deg"
]].isna().mean())

Rows: 670086
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Years: 2009 - 2024
Months: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10)]
Unique tracks: 272892

Positive counts:
6h | box2deg=157 | radius300km=228
12h | box2deg=139 | radius300km=211
24h | box2deg=112 | radius300km=161
48h | box2deg=82 | radius300km=128

NaN fractions:
eta850_abs        0.000000
rh600             0.000000
wshear_200_850    0.000000
sst_K             0.177855
mslp_Pa           0.000000
pi_vmax_ms        0.177856
gpi               0.177856
speed_kmh         0.407249
direction_deg     0.407249
dtype: float64


In [22]:
# ============================================================
# MERGE 300 KM + BOX2DEG LABELS BACK INTO SOFT FEATURE FILE
# Then build 300 km sensitivity case tables
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

LABEL_CSV = SOFT_DIR / "mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"

OUT_MERGED_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt_WITH_LABELS.csv"

R300_CASE_DIR = SOFT_DIR / "exactlead_radius300km_case_tables_soft_2009_2025_JJASO"

LEADS = [6, 12, 24, 48]

# ============================================================
# LOAD FEATURE FILE
# ============================================================

feat = pd.read_csv(FEATURE_CSV, low_memory=False)

feat["time"] = (
    pd.to_datetime(feat["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

print("Loaded feature file")
print("Rows:", len(feat))
print("Columns:", len(feat.columns))
print("Time range:", feat["time"].min(), "to", feat["time"].max())

# ============================================================
# LOAD LABEL FILE
# ============================================================

lab = pd.read_csv(LABEL_CSV, low_memory=False)

lab["time"] = (
    pd.to_datetime(lab["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

label_cols = []

for L in LEADS:
    label_cols += [
        f"box2deg_y_{L}h",
        f"radius300km_y_{L}h"
    ]

needed_label_cols = ["time", "centroid_lat", "centroid_lon"] + label_cols

missing_lab = [c for c in needed_label_cols if c not in lab.columns]

if missing_lab:
    raise ValueError(
        "Missing columns in label file:\n"
        + "\n".join(missing_lab)
    )

lab_small = lab[needed_label_cols].copy()

# Make numeric coordinates consistent
for d in [feat, lab_small]:
    d["centroid_lat"] = pd.to_numeric(d["centroid_lat"], errors="coerce").round(4)
    d["centroid_lon"] = pd.to_numeric(d["centroid_lon"], errors="coerce").round(4)

# Drop duplicate label keys if any
lab_small = lab_small.drop_duplicates(
    subset=["time", "centroid_lat", "centroid_lon"]
)

print("\nLoaded label file")
print("Rows:", len(lab))
print("Label rows after duplicate removal:", len(lab_small))

# ============================================================
# MERGE LABELS INTO FEATURE FILE
# ============================================================

merged = feat.merge(
    lab_small,
    on=["time", "centroid_lat", "centroid_lon"],
    how="left",
    validate="many_to_one"
)

print("\nMerged rows:", len(merged))

# Fill missing labels as 0
for c in label_cols:
    merged[c] = merged[c].fillna(0).astype(int)

# Save merged feature+label file
merged.to_csv(OUT_MERGED_CSV, index=False)

print("\nSaved merged feature + label file:")
print(OUT_MERGED_CSV)

# ============================================================
# 300 KM SENSITIVITY SUMMARY
# ============================================================

summary_rows = []

for L in LEADS:
    box_col = f"box2deg_y_{L}h"
    r300_col = f"radius300km_y_{L}h"

    box = merged[box_col].astype(int)
    r300 = merged[r300_col].astype(int)

    n_box = int(box.sum())
    n_r300 = int(r300.sum())

    summary_rows.append({
        "lead_h": L,
        "box2deg_positive": n_box,
        "radius300km_positive": n_r300,
        "both_positive": int(((box == 1) & (r300 == 1)).sum()),
        "box_only": int(((box == 1) & (r300 == 0)).sum()),
        "radius300km_only": int(((box == 0) & (r300 == 1)).sum()),
        "increase_from_box_to_300km": n_r300 - n_box,
        "increase_percent": ((n_r300 - n_box) / n_box * 100) if n_box > 0 else np.nan
    })

summary = pd.DataFrame(summary_rows)

print("\n300 km sensitivity summary:")
display(summary)

OUT_SUMMARY = SOFT_DIR / "soft_300km_sensitivity_label_summary_2009_2025_JJASO.csv"
summary.to_csv(OUT_SUMMARY, index=False)

print("\nSaved summary:")
print(OUT_SUMMARY)

# ============================================================
# BUILD 300 KM CASE TABLES
# ============================================================

if "build_case_tables_from_master" not in globals():
    raise NameError(
        "build_case_tables_from_master() is not defined. "
        "Run the case-table helper/function cell first."
    )

r300_case_summary = build_case_tables_from_master(
    master_csv=OUT_MERGED_CSV,
    out_case_dir=R300_CASE_DIR,
    dataset_name="soft",
    label_family="radius300km",
    neg_multiplier=5,
    random_state=42,
    drop_missing_predictors=False,
    period_tag="2009_2025_JJASO"
)

print("\n300 km sensitivity case tables created:")
print(R300_CASE_DIR)

display(r300_case_summary)

Loaded feature file
Rows: 718030
Columns: 53
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00

Loaded label file
Rows: 718030
Label rows after duplicate removal: 718030

Merged rows: 718030

Saved merged feature + label file:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt_WITH_LABELS.csv

300 km sensitivity summary:


,lead_h,box2deg_positive,radius300km_positive,both_positive,box_only,radius300km_only,increase_from_box_to_300km,increase_percent
0,6,166,239,166,0,73,73,43.975904
1,12,149,228,149,0,79,79,53.020134
2,24,119,169,119,0,50,50,42.016807
3,48,88,138,88,0,50,50,56.818182



Saved summary:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\soft_300km_sensitivity_label_summary_2009_2025_JJASO.csv


NameError: build_case_tables_from_master() is not defined. Run the case-table helper/function cell first.

CASE TABLES

In [ ]:
# ============================================================
# 300 KM SENSITIVITY CHECK — SOFT DATASET
# Build radius300km case tables + compare with Zhang box2deg
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

SOFT_MASTER_CSV = SOFT_DIR / "mcs_master_soft_exactlead_ready_2009_2025_JJASO.csv"

BOX_CASE_DIR = SOFT_DIR / "exactlead_box2deg_case_tables_soft_2009_2025_JJASO"
R300_CASE_DIR = SOFT_DIR / "exactlead_radius300km_case_tables_soft_2009_2025_JJASO"

LEADS = [6, 12, 24, 48]

# ------------------------------------------------------------
# LOAD MASTER TABLE
# ------------------------------------------------------------
df = pd.read_csv(SOFT_MASTER_CSV, low_memory=False)

df["time"] = (
    pd.to_datetime(df["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

print("=" * 80)
print("SOFT 300 KM SENSITIVITY CHECK")
print("=" * 80)
print("Master file:", SOFT_MASTER_CSV)
print("Rows:", len(df))
print("Time range:", df["time"].min(), "to", df["time"].max())

# ------------------------------------------------------------
# CHECK REQUIRED LABEL COLUMNS
# ------------------------------------------------------------
required_cols = []

for L in LEADS:
    required_cols += [
        f"box2deg_y_{L}h",
        f"radius300km_y_{L}h"
    ]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        "Missing label columns. Make sure your master table includes both "
        "Zhang exact box labels and 300 km sensitivity labels:\n"
        + str(missing)
    )

# ------------------------------------------------------------
# LABEL COMPARISON SUMMARY
# ------------------------------------------------------------
summary_rows = []

for L in LEADS:
    box_col = f"box2deg_y_{L}h"
    r300_col = f"radius300km_y_{L}h"

    box = df[box_col].astype(int)
    r300 = df[r300_col].astype(int)

    n_box = int(box.sum())
    n_r300 = int(r300.sum())

    both_pos = int(((box == 1) & (r300 == 1)).sum())
    box_only = int(((box == 1) & (r300 == 0)).sum())
    r300_only = int(((box == 0) & (r300 == 1)).sum())
    both_neg = int(((box == 0) & (r300 == 0)).sum())

    summary_rows.append({
        "lead_h": L,
        "box2deg_positive": n_box,
        "radius300km_positive": n_r300,
        "both_positive": both_pos,
        "box_only": box_only,
        "radius300km_only": r300_only,
        "both_negative": both_neg,
        "extra_positive_from_300km": r300_only,
        "positive_increase_percent": ((n_r300 - n_box) / n_box * 100) if n_box > 0 else np.nan
    })

summary = pd.DataFrame(summary_rows)

print("\nLabel comparison: box2deg vs radius300km")
display(summary)

OUT_SUMMARY = SOFT_DIR / "soft_label_sensitivity_box2deg_vs_radius300km_2009_2025_JJASO.csv"
summary.to_csv(OUT_SUMMARY, index=False)
print("\nSaved label comparison summary:")
print(OUT_SUMMARY)

# ------------------------------------------------------------
# BUILD 300 KM CASE TABLES
# Requires build_case_tables_from_master() to already be defined
# ------------------------------------------------------------
r300_case_summary = build_case_tables_from_master(
    master_csv=SOFT_MASTER_CSV,
    out_case_dir=R300_CASE_DIR,
    dataset_name="soft",
    label_family="radius300km",
    neg_multiplier=5,
    random_state=42,
    drop_missing_predictors=False,
    period_tag="2009_2025_JJASO"
)

print("\n300 km sensitivity case tables created.")
print("Output directory:")
print(R300_CASE_DIR)

display(r300_case_summary)

In [ ]:
# ============================================================
# BUILD LEAD-SPECIFIC CASE TABLES — SOFT 2009–2024
# master table -> case tables for ML
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

LEADS = [6, 12, 24, 48]

ML_FEATURE_COLS = [
    "lon", "lat", "month",
    "area_km2", "tb_mean", "tb_min",
    "speed_kmh", "direction_deg",
    "eta850_abs", "rh600", "wshear_200_850",
    "pi_vmax_ms", "gpi"
]

ID_COLS = [
    "track_id", "time",
    "centroid_lon", "centroid_lat"
]


def build_case_tables_from_master(
    master_csv,
    out_case_dir,
    dataset_name="soft",
    label_family="box2deg",
    neg_multiplier=5,
    random_state=42,
    drop_missing_predictors=False,
    period_tag="2009_2024_JJASO"
):
    master_csv = Path(master_csv)
    out_case_dir = Path(out_case_dir)
    out_case_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"BUILD CASE TABLES: {dataset_name.upper()} | LABEL={label_family}")
    print("=" * 70)

    if not master_csv.exists():
        raise FileNotFoundError(master_csv)

    df = pd.read_csv(master_csv)
    df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

    required_cols = ID_COLS + ML_FEATURE_COLS
    label_cols = [f"{label_family}_y_{L}h" for L in LEADS]

    missing = [c for c in required_cols + label_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns:\n{missing}")

    print("Master rows:", len(df))
    print("Time range:", df["time"].min(), "to", df["time"].max())

    before = len(df)

    if drop_missing_predictors:
        df = df.dropna(subset=ML_FEATURE_COLS).copy()

    after = len(df)

    print("Rows before predictor NaN drop:", before)
    print("Rows after predictor NaN drop:", after)
    print("Dropped:", before - after)

    # ------------------------------------------------------------
    # Never-positive negative pool
    # ------------------------------------------------------------
    any_positive = np.zeros(len(df), dtype=bool)

    for L in LEADS:
        any_positive |= (df[f"{label_family}_y_{L}h"].astype(int).values == 1)

    neg_pool = df.loc[~any_positive].copy()

    print("Never-positive negative pool:", len(neg_pool))

    summary_rows = []

    for L in LEADS:
        label_col = f"{label_family}_y_{L}h"

        pos = df[df[label_col].astype(int) == 1].copy()
        pos["lead_h"] = L
        pos["y"] = 1

        n_pos = len(pos)

        if n_pos == 0:
            print(f"[WARN] {L}h has zero positives. Skipping.")
            continue

        n_neg_target = neg_multiplier * n_pos

        if len(neg_pool) < n_neg_target:
            print(f"[WARN] Not enough negatives for {L}h. Using all available negatives.")
            neg = neg_pool.copy()
        else:
            neg = neg_pool.sample(n=n_neg_target, random_state=random_state + L).copy()

        neg["lead_h"] = L
        neg["y"] = 0

        keep_cols = ID_COLS + ML_FEATURE_COLS + ["lead_h", "y"]

        case = pd.concat(
            [pos[keep_cols], neg[keep_cols]],
            ignore_index=True
        )

        case = case.sample(frac=1, random_state=random_state + L).reset_index(drop=True)

        out_csv = out_case_dir / f"{label_family}_case_table_{dataset_name}_{L}h.csv"
        case.to_csv(out_csv, index=False)

        n_pos_final = int((case["y"] == 1).sum())
        n_neg_final = int((case["y"] == 0).sum())

        summary_rows.append({
            "dataset": dataset_name,
            "label_family": label_family,
            "lead_h": L,
            "positive": n_pos_final,
            "negative": n_neg_final,
            "total": len(case),
            "pos_frac": n_pos_final / len(case)
        })

        print("\n[SAVED]", out_csv.name)
        print("Positive:", n_pos_final)
        print("Negative:", n_neg_final)
        print("Total:", len(case))
        print("Pos frac:", n_pos_final / len(case))

    summary = pd.DataFrame(summary_rows)

    summary_csv = out_case_dir / f"case_table_summary_{dataset_name}_{label_family}_{period_tag}.csv"
    summary.to_csv(summary_csv, index=False)

    print("\n[SAVED SUMMARY]")
    print(summary_csv)
    print(summary)

    return summary

In [ ]:
SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

SOFT_MASTER_CSV = SOFT_DIR / "mcs_master_soft_exactlead_ready_2009_2024_JJASO.csv"

SOFT_CASE_DIR = SOFT_DIR / "exactlead_box2deg_case_tables_soft_2009_2024_JJASO"

soft_case_summary = build_case_tables_from_master(
    master_csv=SOFT_MASTER_CSV,
    out_case_dir=SOFT_CASE_DIR,
    dataset_name="soft",
    label_family="box2deg",
    neg_multiplier=5,
    random_state=42,
    drop_missing_predictors=False,
    period_tag="2009_2024_JJASO"
)


BUILD CASE TABLES: SOFT | LABEL=box2deg
Master rows: 670086
Time range: 2009-06-01 00:00:00 to 2024-10-31 21:00:00
Rows before predictor NaN drop: 670086
Rows after predictor NaN drop: 670086
Dropped: 0
Never-positive negative pool: 669596

[SAVED] box2deg_case_table_soft_6h.csv
Positive: 157
Negative: 785
Total: 942
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_soft_12h.csv
Positive: 139
Negative: 695
Total: 834
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_soft_24h.csv
Positive: 112
Negative: 560
Total: 672
Pos frac: 0.16666666666666666

[SAVED] box2deg_case_table_soft_48h.csv
Positive: 82
Negative: 410
Total: 492
Pos frac: 0.16666666666666666

[SAVED SUMMARY]
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_soft_2009_2024_JJASO\case_table_summary_soft_box2deg_2009_2024_JJASO.csv
  dataset label_family  lead_h  positive  negative  total  pos_frac
0    soft      box2deg       6       157       785    942  0.166667
1    soft   

In [ ]:
from pathlib import Path
import pandas as pd

CASE_DIR = Path(
    r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\exactlead_box2deg_case_tables_soft_2009_2024_JJASO"
)

for L in [6, 12, 24, 48]:
    fp = CASE_DIR / f"box2deg_case_table_soft_{L}h.csv"
    df = pd.read_csv(fp)
    print(f"\nLead {L}h")
    print("exists:", fp.exists())
    print("rows:", len(df))
    print("positive:", int(df["y"].sum()))
    print("negative:", int((df["y"] == 0).sum()))
    print("NaN predictor rows:", df[[
        "lon", "lat", "month",
        "area_km2", "tb_mean", "tb_min",
        "speed_kmh", "direction_deg",
        "eta850_abs", "rh600", "wshear_200_850",
        "pi_vmax_ms", "gpi"
    ]].isna().any(axis=1).sum())


Lead 6h
exists: True
rows: 942
positive: 157
negative: 785
NaN predictor rows: 471

Lead 12h
exists: True
rows: 834
positive: 139
negative: 695
NaN predictor rows: 410

Lead 24h
exists: True
rows: 672
positive: 112
negative: 560
NaN predictor rows: 357

Lead 48h
exists: True
rows: 492
positive: 82
negative: 410
NaN predictor rows: 264


In [ ]:
# ============================================================
# NEW/SOFT FEATURE EXTRACTION — 2025 ONLY
# Uses updated 2009–2025 label file with corrected IBTrACS genesis
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

MCS_LABELED_CSV = SOFT_DIR / "mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"

# Same folder where 2009–2024 monthly features already exist
FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"
FEATURE_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

ERR_DIR = FEATURE_MONTHLY_DIR / "errors"
ERR_DIR.mkdir(parents=True, exist_ok=True)

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

YEARS_2025_ONLY = [2025]
MONTHS = [6, 7, 8, 9, 10]

# ============================================================
# CHECK FUNCTION EXISTS
# ============================================================

if "extract_features_one_month" not in globals():
    raise NameError(
        "extract_features_one_month() is not defined in this notebook. "
        "Run the feature-extraction helper/function cells first."
    )

# ============================================================
# LOAD UPDATED NEW/SOFT LABEL FILE
# ============================================================

if not MCS_LABELED_CSV.exists():
    raise FileNotFoundError(MCS_LABELED_CSV)

mcs_all = pd.read_csv(MCS_LABELED_CSV, low_memory=False)

mcs_all["time"] = (
    pd.to_datetime(mcs_all["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

mcs_all = mcs_all.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

mcs_all = mcs_all[
    mcs_all["time"].dt.year.between(2009, 2025)
    & mcs_all["time"].dt.month.isin(MONTHS)
].copy()

mcs_all["month"] = mcs_all["time"].dt.month
mcs_all["lat"] = pd.to_numeric(mcs_all["centroid_lat"], errors="coerce")
mcs_all["lon"] = pd.to_numeric(mcs_all["centroid_lon"], errors="coerce")

print("Loaded updated NEW/SOFT label file")
print("Rows:", len(mcs_all))
print("Time range:", mcs_all["time"].min(), "to", mcs_all["time"].max())
print("2025 rows:", len(mcs_all[mcs_all["time"].dt.year == 2025]))

print("\n2025 positive counts:")
m25 = mcs_all[mcs_all["time"].dt.year == 2025]
for L in [6, 12, 24, 48]:
    print(
        f"{L}h | "
        f"box2deg={int(m25[f'box2deg_y_{L}h'].sum())} | "
        f"radius300km={int(m25[f'radius300km_y_{L}h'].sum())}"
    )

# ============================================================
# RUN 2025 ONLY
# ============================================================

done_2025 = []

for y in YEARS_2025_ONLY:
    for m in MONTHS:
        print("\n" + "=" * 70)
        print(f"EXTRACTING NEW/SOFT FEATURES: {y}-{m:02d}")
        print("=" * 70)

        out_fp = extract_features_one_month(
            year=y,
            month=m,
            overwrite=False
        )

        if out_fp is not None:
            done_2025.append(out_fp)

print("\n2025 monthly feature files available:", len(done_2025))
for fp in done_2025:
    print(fp)

Loaded updated NEW/SOFT label file
Rows: 718030
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00
2025 rows: 47944

2025 positive counts:
6h | box2deg=13 | radius300km=16
12h | box2deg=11 | radius300km=18
24h | box2deg=9 | radius300km=12
48h | box2deg=7 | radius300km=11

EXTRACTING NEW/SOFT FEATURES: 2025-06
[RESUME/REUSE] mcs_features_PI_GPI_soft_202506.csv
Existing rows: 8987

202506
Rows: 8987
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202506.csv

EXTRACTING NEW/SOFT FEATURES: 2025-07
[RESUME/REUSE] mcs_features_PI_GPI_soft_202507.csv
Existing rows: 9887

202507
Rows: 9887
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202507.csv

EXTRACTING NEW/SOFT FEATURES: 2025-08
[RESUME/REUSE] mcs_features_PI_GPI_soft_202508.csv
Existing rows: 10541

202508
Rows: 10541
Rows still to attempt: 0
[DONE/REUSED] mcs_features_PI_GPI_soft_202508.csv

EXTRACTING NEW/SOFT FEATURES: 2025-09
[RESUME/REUSE] mcs_features_PI_GPI_soft_202509.csv
Existing rows: 9935


In [ ]:
# ============================================================
# AUDIT NEW/SOFT MONTHLY FEATURES — EXPECT 85 FILES
# ============================================================

from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")
FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"

expected = [
    FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_{y}{m:02d}.csv"
    for y in range(2009, 2026)
    for m in [6, 7, 8, 9, 10]
]

missing = [fp for fp in expected if not fp.exists()]

print("Expected monthly feature files:", len(expected))
print("Missing:", len(missing))

if missing:
    print("\nMissing files:")
    for fp in missing:
        print(fp.name)
else:
    print("All 85 monthly NEW/SOFT feature files exist.")

# Quick 2025 check
for m in [6, 7, 8, 9, 10]:
    fp = FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_2025{m:02d}.csv"
    if fp.exists():
        df = pd.read_csv(fp, low_memory=False)
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)
        print("\n", fp.name)
        print("rows:", len(df))
        print("time:", df["time"].min(), "to", df["time"].max())
        print("gpi NaN frac:", df["gpi"].isna().mean() if "gpi" in df.columns else "no gpi column")

Expected monthly feature files: 85
Missing: 0
All 85 monthly NEW/SOFT feature files exist.

 mcs_features_PI_GPI_soft_202506.csv
rows: 8987
time: 2025-06-01 00:00:00 to 2025-06-30 21:00:00
gpi NaN frac: 1.0

 mcs_features_PI_GPI_soft_202507.csv
rows: 9887
time: 2025-07-01 00:00:00 to 2025-07-31 21:00:00
gpi NaN frac: 1.0

 mcs_features_PI_GPI_soft_202508.csv
rows: 10541
time: 2025-08-01 00:00:00 to 2025-08-31 21:00:00
gpi NaN frac: 1.0

 mcs_features_PI_GPI_soft_202509.csv
rows: 9935
time: 2025-09-01 00:00:00 to 2025-09-30 21:00:00
gpi NaN frac: 1.0

 mcs_features_PI_GPI_soft_202510.csv
rows: 8594
time: 2025-10-01 00:00:00 to 2025-10-31 21:00:00
gpi NaN frac: 1.0


In [ ]:
# ============================================================
# CONCATENATE NEW/SOFT FULL FEATURE FILE — 2009–2025
# ============================================================

from pathlib import Path
import pandas as pd

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")
FEATURE_MONTHLY_DIR = SOFT_DIR / "features_monthly_soft_2009_2024"

OUT_FULL_FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

feature_files = [
    FEATURE_MONTHLY_DIR / f"mcs_features_PI_GPI_soft_{y}{m:02d}.csv"
    for y in range(2009, 2026)
    for m in [6, 7, 8, 9, 10]
]

missing = [fp for fp in feature_files if not fp.exists()]

print("Expected:", len(feature_files))
print("Missing:", len(missing))

if missing:
    for fp in missing:
        print(fp.name)
    raise FileNotFoundError("Some monthly feature files are missing.")

dfs = []

for fp in feature_files:
    print("reading:", fp.name)
    dfs.append(pd.read_csv(fp, low_memory=False))

features = pd.concat(dfs, ignore_index=True)

features["time"] = (
    pd.to_datetime(features["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

features = (
    features
    .sort_values(["time", "track_id"])
    .reset_index(drop=True)
)

features.to_csv(OUT_FULL_FEATURE_CSV, index=False)

print("\nSaved full NEW/SOFT feature file:")
print(OUT_FULL_FEATURE_CSV)

print("\nQC:")
print("Rows:", len(features))
print("Time range:", features["time"].min(), "to", features["time"].max())
print("Years:", features["time"].dt.year.min(), "-", features["time"].dt.year.max())
print("Months:", sorted(features["time"].dt.month.unique()))

print("\nNaN fractions:")
print(features[[
    "eta850_abs", "rh600", "wshear_200_850",
    "sst_K", "mslp_Pa", "pi_vmax_ms", "gpi",
    "speed_kmh", "direction_deg"
]].isna().mean())

Expected: 85
Missing: 0
reading: mcs_features_PI_GPI_soft_200906.csv
reading: mcs_features_PI_GPI_soft_200907.csv
reading: mcs_features_PI_GPI_soft_200908.csv
reading: mcs_features_PI_GPI_soft_200909.csv
reading: mcs_features_PI_GPI_soft_200910.csv
reading: mcs_features_PI_GPI_soft_201006.csv
reading: mcs_features_PI_GPI_soft_201007.csv
reading: mcs_features_PI_GPI_soft_201008.csv
reading: mcs_features_PI_GPI_soft_201009.csv
reading: mcs_features_PI_GPI_soft_201010.csv
reading: mcs_features_PI_GPI_soft_201106.csv
reading: mcs_features_PI_GPI_soft_201107.csv
reading: mcs_features_PI_GPI_soft_201108.csv
reading: mcs_features_PI_GPI_soft_201109.csv
reading: mcs_features_PI_GPI_soft_201110.csv
reading: mcs_features_PI_GPI_soft_201206.csv
reading: mcs_features_PI_GPI_soft_201207.csv
reading: mcs_features_PI_GPI_soft_201208.csv
reading: mcs_features_PI_GPI_soft_201209.csv
reading: mcs_features_PI_GPI_soft_201210.csv
reading: mcs_features_PI_GPI_soft_201306.csv
reading: mcs_features_PI_GPI_so

In [ ]:
# ============================================================
# MERGE 300 KM + BOX2DEG LABELS BACK INTO SOFT FEATURE FILE
# Then build 300 km sensitivity case tables
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

SOFT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

FEATURE_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt.csv"

LABEL_CSV = SOFT_DIR / "mcs_labeled_final_new_2009_2025_JJASO_exactlead_box2deg_radius300km_UPDATED.csv"

OUT_MERGED_CSV = SOFT_DIR / "mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt_WITH_LABELS.csv"

R300_CASE_DIR = SOFT_DIR / "exactlead_radius300km_case_tables_soft_2009_2025_JJASO"

LEADS = [6, 12, 24, 48]

# ============================================================
# LOAD FEATURE FILE
# ============================================================

feat = pd.read_csv(FEATURE_CSV, low_memory=False)

feat["time"] = (
    pd.to_datetime(feat["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

print("Loaded feature file")
print("Rows:", len(feat))
print("Columns:", len(feat.columns))
print("Time range:", feat["time"].min(), "to", feat["time"].max())

# ============================================================
# LOAD LABEL FILE
# ============================================================

lab = pd.read_csv(LABEL_CSV, low_memory=False)

lab["time"] = (
    pd.to_datetime(lab["time"], errors="coerce", utc=True)
    .dt.tz_convert(None)
)

label_cols = []

for L in LEADS:
    label_cols += [
        f"box2deg_y_{L}h",
        f"radius300km_y_{L}h"
    ]

needed_label_cols = ["time", "centroid_lat", "centroid_lon"] + label_cols

missing_lab = [c for c in needed_label_cols if c not in lab.columns]

if missing_lab:
    raise ValueError(
        "Missing columns in label file:\n"
        + "\n".join(missing_lab)
    )

lab_small = lab[needed_label_cols].copy()

# Make numeric coordinates consistent
for d in [feat, lab_small]:
    d["centroid_lat"] = pd.to_numeric(d["centroid_lat"], errors="coerce").round(4)
    d["centroid_lon"] = pd.to_numeric(d["centroid_lon"], errors="coerce").round(4)

# Drop duplicate label keys if any
lab_small = lab_small.drop_duplicates(
    subset=["time", "centroid_lat", "centroid_lon"]
)

print("\nLoaded label file")
print("Rows:", len(lab))
print("Label rows after duplicate removal:", len(lab_small))

# ============================================================
# MERGE LABELS INTO FEATURE FILE
# ============================================================

merged = feat.merge(
    lab_small,
    on=["time", "centroid_lat", "centroid_lon"],
    how="left",
    validate="many_to_one"
)

print("\nMerged rows:", len(merged))

# Fill missing labels as 0
for c in label_cols:
    merged[c] = merged[c].fillna(0).astype(int)

# Save merged feature+label file
merged.to_csv(OUT_MERGED_CSV, index=False)

print("\nSaved merged feature + label file:")
print(OUT_MERGED_CSV)

# ============================================================
# 300 KM SENSITIVITY SUMMARY
# ============================================================

summary_rows = []

for L in LEADS:
    box_col = f"box2deg_y_{L}h"
    r300_col = f"radius300km_y_{L}h"

    box = merged[box_col].astype(int)
    r300 = merged[r300_col].astype(int)

    n_box = int(box.sum())
    n_r300 = int(r300.sum())

    summary_rows.append({
        "lead_h": L,
        "box2deg_positive": n_box,
        "radius300km_positive": n_r300,
        "both_positive": int(((box == 1) & (r300 == 1)).sum()),
        "box_only": int(((box == 1) & (r300 == 0)).sum()),
        "radius300km_only": int(((box == 0) & (r300 == 1)).sum()),
        "increase_from_box_to_300km": n_r300 - n_box,
        "increase_percent": ((n_r300 - n_box) / n_box * 100) if n_box > 0 else np.nan
    })

summary = pd.DataFrame(summary_rows)

print("\n300 km sensitivity summary:")
display(summary)

OUT_SUMMARY = SOFT_DIR / "soft_300km_sensitivity_label_summary_2009_2025_JJASO.csv"
summary.to_csv(OUT_SUMMARY, index=False)

print("\nSaved summary:")
print(OUT_SUMMARY)

# ============================================================
# BUILD 300 KM CASE TABLES
# ============================================================

if "build_case_tables_from_master" not in globals():
    raise NameError(
        "build_case_tables_from_master() is not defined. "
        "Run the case-table helper/function cell first."
    )

r300_case_summary = build_case_tables_from_master(
    master_csv=OUT_MERGED_CSV,
    out_case_dir=R300_CASE_DIR,
    dataset_name="soft",
    label_family="radius300km",
    neg_multiplier=5,
    random_state=42,
    drop_missing_predictors=False,
    period_tag="2009_2025_JJASO"
)

print("\n300 km sensitivity case tables created:")
print(R300_CASE_DIR)

display(r300_case_summary)

Loaded feature file
Rows: 718030
Columns: 53
Time range: 2009-06-01 00:00:00 to 2025-10-31 21:00:00

Loaded label file
Rows: 718030
Label rows after duplicate removal: 718030

Merged rows: 718030

Saved merged feature + label file:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\mcs_features_PI_GPI_soft_2009_2025_JJASO_zhang4pt_WITH_LABELS.csv

300 km sensitivity summary:


,lead_h,box2deg_positive,radius300km_positive,both_positive,box_only,radius300km_only,increase_from_box_to_300km,increase_percent
0,6,166,239,166,0,73,73,43.975904
1,12,149,228,149,0,79,79,53.020134
2,24,119,169,119,0,50,50,42.016807
3,48,88,138,88,0,50,50,56.818182



Saved summary:
C:\Users\_s2218026\Documents\lab\mcs_output_softtest\soft_300km_sensitivity_label_summary_2009_2025_JJASO.csv


NameError: build_case_tables_from_master() is not defined. Run the case-table helper/function cell first.